# Clustering

Functions that fit a configuration and return its clusters. All k-medoids configurations go through
`SampleDistClustering` with `frac_sample_size = 1`: with 216 patients there is no need to subsample, so the
distance matrix is computed on all patients and PAM is run on it. The only k-means configuration is C1-b.

Imports

In [ ]:
import numpy as np
from kmedoids import KMedoids
from sklearn.cluster import KMeans
from db_robust_clust.models import SampleDistClustering
from robust_mixed_dist.mixed import related_metric_scaling_dist_matrix

### Exact RelMS

`SampleDistClustering(metric='relms')` uses `related_metric_scaling_dist_matrix_faster` internally, an
approximation meant for large samples. With 216 patients the exact version is affordable, so this small subclass
replaces only that call with `related_metric_scaling_dist_matrix`. For G-Gower it behaves exactly like the original
class.

In [ ]:
class ExactRelMSSampleDistClustering(SampleDistClustering):
    """SampleDistClustering with the exact RelMS distance.

    The library maps metric='relms' to related_metric_scaling_dist_matrix_faster, an
    approximation intended for large samples. With frac_sample_size=1 the class builds a
    single distance matrix on all units and passes it to PAM, so swapping in the exact
    function here is all that is needed.
    """

    def _compute_dist_matrix(self, X):
        if self.metric == "relms":
            return related_metric_scaling_dist_matrix(
                X, self.p1, self.p2, self.p3, self.d1, self.d2, self.d3,
                q=self.q, robust_method=self.robust_method, alpha=self.alpha,
                weights=self.weights, return_combined_distances=True,
            )
        return super()._compute_dist_matrix(X)

### PAM

PAM with `init='build'`, 100 iterations and a fixed seed, on a precomputed distance matrix.

In [ ]:
def make_pam(k, random_state=cfg.RANDOM_STATE):
    return KMedoids(n_clusters=k, metric="precomputed", random_state=random_state, **cfg.PAM)

### Bootstrap refits for the stability check

Each refit draws 216 patients with replacement, refits the whole configuration on them (distance, robust
covariance, medoids or centroids) and then assigns all 216 original patients to the nearest medoid or centroid.
Resampling is used instead of changing the seed because, without subsampling, a different seed gives the same
partition.

In [ ]:
def bootstrap_partitions(name, k, df, n_refits=cfg.N_STABILITY_REFITS, metric=None):
    """Partitions of all patients from n_refits fits on bootstrap resamples (refit r uses seed r).

    Everything is re-estimated on each resample (distance, robust covariance, block
    variabilities, medoids or centroids); every original patient is then assigned to its
    nearest medoid/centroid. For RelMS configurations the library assigns new patients with
    the G-Gower point distance, so only G-Gower and k-means configurations are used here.
    """
    config = cfg.CONFIGS[name]
    n = len(df)
    partitions = []
    for r in range(n_refits):
        idx = np.random.default_rng(r).integers(0, n, n)
        if config["algorithm"] == "kmeans":
            X = build_kmeans_matrix(df, config)["X"]
            model = KMeans(n_clusters=k, random_state=cfg.RANDOM_STATE, n_init=cfg.KMEANS_N_INIT).fit(X[idx])
            partitions.append(model.predict(X))
            continue
        design = build_design(df, config)
        model = ExactRelMSSampleDistClustering(
            clustering_method=make_pam(k), metric=metric or config["metric"],
            frac_sample_size=cfg.FRAC_SAMPLE_SIZE, random_state=cfg.RANDOM_STATE,
            p1=design["p1"], p2=design["p2"], p3=design["p3"],
            d1=design["d1"], d2=design["d2"], d3=design["d3"],
            robust_method=cfg.ROBUST_METHOD, alpha=cfg.ALPHA,
        ).fit(design["X"][idx])
        partitions.append(np.asarray(model.predict(design["X"])).astype(int))
    return np.array(partitions)

### Fitting one configuration

- C1-b: k-means with 10 starts on the standardised clinical variables.
- All others: block layout from `build_design`, then `SampleDistClustering` with G-Gower or exact RelMS
  (robust Mahalanobis inside the clinical block) and PAM.
- M3 takes `metric=` as an argument because its distance is chosen after comparing M1 and M2.

In [ ]:
def fit_configuration(name, k, df, metric=None, random_state=cfg.RANDOM_STATE):
    """Partition of df under configuration `name` with k clusters.

    `metric` is only needed for M3, whose distance is the better of M1/M2.
    Returns labels (0..k-1), medoid row indices (None for k-means) and the
    distance matrix the partition was obtained in (None for k-means).
    """
    config = cfg.CONFIGS[name]

    if config["algorithm"] == "kmeans":
        design = build_kmeans_matrix(df, config)
        model = KMeans(n_clusters=k, random_state=random_state, n_init=cfg.KMEANS_N_INIT).fit(design["X"])
        return {"config": name, "k": k, "labels": model.labels_.astype(int),
                "medoids": None, "D": None, "X": design["X"], "model": model}

    metric = metric or config["metric"]
    if metric is None:
        raise ValueError(f"{name}: metric must be given (set after comparing M1 and M2)")

    design = build_design(df, config)
    model = ExactRelMSSampleDistClustering(
        clustering_method=make_pam(k, random_state),
        metric=metric,
        frac_sample_size=cfg.FRAC_SAMPLE_SIZE,
        random_state=random_state,
        p1=design["p1"], p2=design["p2"], p3=design["p3"],
        d1=design["d1"], d2=design["d2"], d3=design["d3"],
        robust_method=cfg.ROBUST_METHOD, alpha=cfg.ALPHA,
    ).fit(design["X"])

    D = model.dist_output[0] if isinstance(model.dist_output, tuple) else model.dist_output
    return {"config": name, "k": k, "metric": metric,
            "labels": np.asarray(model.labels_).astype(int),
            "medoids": np.asarray(model.medoid_indices_).astype(int),
            "D": np.asarray(D), "X": design["X"], "model": model}